In [1]:
import pandas as pd
import os
import geopandas as gpd
import pickle

## Collect Linking Tool Results

* **Define the Linking Tool Results Directory:** Assuming you have the following standard file structure:

    ```
    repository
        |___Bidirectional_Linking_Tool
            |___results
        |___BC-CLEWS-Model
            |__from_linking_tool
                |__results
    ```

    - `repository`: All Git Repositories' Root directory.
    - `Bidirectional_Linking_Tool`: Directory containing the _Bidirectional Linking Tool_.
        - `results`: Subdirectory within *Bidirectional_Linking_Tool*.
    - `BC-CLEWS-Model`: Directory containing the _BC-CLEWS-Model_ git repo.
        - `from_linking_tool`: Subdirectory within _BC-CLEWS-Model_.
            - `results`: Subdirectory within *from_linking_tool* , contains the pickle files from Linking tool.

In [2]:
linking_tool_directs={
    'root': 'Bidirectional_Linking_Tool',
    'results': 'results',
}

bc_nexus_direct={
    'root': 'BC-CLEWS-Model',
    'collect_linking_tool_results':'from_linking_tool',
    'results': 'results',
}

print(f"Please make sure you have '{linking_tool_directs['root']}' directory present in your 'repository' root.")

Please make sure you have 'Bidirectional_Linking_Tool' directory present in your 'repository' root.


* Bash Cmd

__!!! Caution__: The Bash cmd execution relies on the aforemention __standard Folder Structure__

> copy-paste result pickle files (__cluster info__ & __timeseries of clusters__) from Linking tool to BC-Nexus directories for further analysis.

In [3]:
!cd .. && cd .. && mkdir -p {bc_nexus_direct['root']}/{bc_nexus_direct['collect_linking_tool_results']}/{bc_nexus_direct['results']} && cp {linking_tool_directs['root']}/{linking_tool_directs['results']}/*.pkl {bc_nexus_direct['root']}/{bc_nexus_direct['collect_linking_tool_results']}/{bc_nexus_direct['results']}

## Define Timeslice conversion results Directory

In [4]:
directories = 'new_sites_timeslice','new_sites'

for directory in directories:
    # Check if the directory exists, and create it if it doesn't
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Directory '{directory}' created successfully.")
    else:
        print(f"Directory '{directory}' already exists.")

Directory 'new_sites_timeslice' already exists.
Directory 'new_sites' already exists.


# Load Timelsice Names

In [5]:
import pandas as pd
TIMESLICES_df=pd.read_csv('/localhome/mei3/eliasinul/repositories/BC-CLEWS-Model/datapackage/SETS/CLEWsy_outputs/TIMESLICE.csv')
TIMESLICES=TIMESLICES_df.VALUE.to_list()

TECHNOLOGIES_df=pd.read_csv('/localhome/mei3/eliasinul/repositories/BC-CLEWS-Model/datapackage/SETS/CLEWsy_outputs/TECHNOLOGY.csv')
seen_TECHNOLOGIES=TECHNOLOGIES_df.VALUE.to_list()

# Define Season and Timeslice Configuration

> To be replaced by __TSA (Time Slice Aggregation)__ optimization

In [6]:
season_config = {
    "Winter1": {"start": "2021-01-01", "end": "2021-03-19", "daytime": {"day": 8, "night": 17}},
    "Spring": {"start": "2021-03-20", "end": "2021-06-19", "daytime": {"day": 6, "night": 20}},
    "Summer": {"start": "2021-06-20", "end": "2021-09-21", "daytime": {"day": 6, "night": 21}},
    "Fall": {"start": "2021-09-22", "end": "2021-12-20", "daytime": {"day": 8, "night": 18}},
    "Winter2": {"start": "2021-12-21", "end": "2021-12-31", "daytime": {"day": 8, "night": 17}}
}

# Mapping dictionary
timeslice_name_mapping = {
    'Fall_D': 'SEA2D',
    'Fall_N': 'SEA2N',
    'Out_of_Season': 'NA',
    'Spring_D': 'SEA3D',
    'Spring_N': 'SEA3N',
    'Summer_D': 'SEA4D',
    'Summer_N': 'SEA4N',
    'Winter1_D': 'SEA1D',
    'Winter1_N': 'SEA1N',
    'Winter2_D': 'SEA1D',
    'Winter2_N': 'SEA1N'
}

# Define Funcs

> to be automated further inline with timeslice configuration for __BC-Nexus__

In [7]:
def create_timeslice_from_cluster(cluster_ts_pickle,site_name,season_config,timeslice_name_mapping):

    df_ts=pd.read_pickle(cluster_ts_pickle)

    # Convert season_config dates to datetime with error handling
    for season, dates in season_config.items():
        try:
            dates["start"] = pd.to_datetime(dates["start"])
            dates["end"] = pd.to_datetime(dates["end"])
        except ValueError as e:
            print(f"Error converting dates for season '{season}': {e}")

    # Improved function for clarity and error handling
    def get_season_time_of_day(datetime):
        for season, dates in season_config.items():
            if dates["start"] <= datetime <= dates["end"]:
                # Handle specific time ranges for day and night
                if 0 <= datetime.hour <= dates["daytime"]["day"]:
                    return f"{season}_N"  # Night
                elif dates["daytime"]["day"] <= datetime.hour <= dates["daytime"]["night"]:
                    return f"{season}_D"  # Day
                else:  # Outside daytime range
                    return f"{season}_N"  # Assuming night for values outside day/night times
        # Handle date outside any season (optional: return specific value or raise an error)
        return "Out_of_Season"  # Example handling

    # Create the 'season_daytime' column with error handling
    try:
        df_ts['season_daytime'] = df_ts.index.map(get_season_time_of_day)
    except TypeError as e:
        print("Error applying function: ", e)

    # Display the DataFrame
    # Group by season and day/night
    grouped_df = df_ts.groupby(['season_daytime']).agg({
        site_name: ['mean']  # You can add more aggregation functions here e.g 'std'
    })

    # Display the grouped DataFrame
    grouped_df['CF_mean'] = grouped_df[site_name]['mean']
    grouped_df['TIMESLICE'] = grouped_df.index.map(timeslice_name_mapping)
    grouped_df=grouped_df.groupby('TIMESLICE').mean() 
    
    grouped_df.columns=grouped_df.columns.get_level_values(0)
    return grouped_df,df_ts

In [8]:
def create_n_save_timeslice_files(ts_pickle,cluster_pickle,asset_type,season_config,timeslice_name_mapping):
    sites_df=gpd.GeoDataFrame(pd.read_pickle(cluster_pickle))
    sites_without_geom=sites_df.drop(columns=['geometry'])
    sites_without_geom.to_csv(f'new_sites/{asset_type}.csv')
    
    sites=list(sites_df.index)
    cluster_ts_timeslice = {}
    cluster_ts={}

    for site in sites:
        cluster_ts_timeslice[site],cluster_ts = create_timeslice_from_cluster(ts_pickle, site,season_config,timeslice_name_mapping)
        cluster_ts_timeslice[site].to_csv(f'new_sites_timeslice/{asset_type}_{site}.csv')
        
    
    print(f"BCNexus compatible timeseries created for - {asset_type} Clusters")
    return cluster_ts,cluster_ts_timeslice,sites_df

# Convert and Extract Timeslice Downscaling Results

In [9]:
solar_ts,solar_timeslices,solar_sites=create_n_save_timeslice_files('results/Solar_Top_Sites_Clustered_CF_timeseries.pkl','results/Solar_Top_Sites_Clustered.pkl','Solar',season_config,timeslice_name_mapping)
wind_ts,wind_timeslices,wind_sites=create_n_save_timeslice_files('results/Wind_Top_Sites_Clustered_CF_timeseries.pkl','results/Wind_Top_Sites_Clustered.pkl','Wind',season_config,timeslice_name_mapping)

BCNexus compatible timeseries created for - Solar Clusters
BCNexus compatible timeseries created for - Wind Clusters


# Define the list of years and planning horizon

In [10]:
sites_COD = 2030  # Define the starting year
planning_horizon = 2050
operational_years_count = planning_horizon - sites_COD + 1
years = list(range(sites_COD, planning_horizon + 1))

# Process the Results and plug-in other required Datafields

In [11]:
import pandas as pd
from typing import Dict, List, Set

def process_timeslices_forNexus(
        downscaled_timeslice_dict: Dict[str, pd.DataFrame],
        TIMESLICES: List[str],
        seen_TECHNOLOGIES: Set[str],
        technology_prefix: str,
        site_no_start: int,
        sites_COD: int,
        planning_horizon: int = 2050,
        region: str = 'REGION1'
    ) -> Dict[str, pd.DataFrame]:
    """
    Processes timeslice data for Nexus compatibility.

    Parameters:
    downscaled_timeslice_dict (dict): Dictionary of downscaled timeslice dataframes.
    TIMESLICES (list): List of valid timeslices.
    seen_TECHNOLOGIES (set): Set of seen technology names to check for duplicates.
    technology_prefix (str): Prefix for the technology names.
    site_no_start (int): Starting number for the site numbering.
    sites_COD (int): Start year for the sites.
    planning_horizon (int): End year for the planning horizon.
    region (str): Region name to be added to the data.

    Returns:
    dict: Dictionary of processed dataframes for Nexus data intake validation.
    """

    _nexus_timeslices_ = {}
    years = list(range(sites_COD, planning_horizon + 1))
    operational_years_count = planning_horizon - sites_COD + 1

    # Process each site in the timeslice dictionary
    for site_sl, (site, site_ts_df) in enumerate(downscaled_timeslice_dict.items(), start=1):
        # Reset the index to turn 'timeslices' into a regular column
        site_ts_df=site_ts_df.reset_index()
        # Select only the 'TIMESLICE' and 'CF_mean' columns
        site_ts_df = site_ts_df[['TIMESLICE', 'CF_mean']]

        # Filter data based on TIMESLICES
        _clean_data_ = site_ts_df[site_ts_df['TIMESLICE'].isin(TIMESLICES)]
        _clean_data_ = _clean_data_.rename(columns={'CF_mean': 'VALUE'})

        # Repeat each row for each year in the operational period
        _all_datafields_ = pd.concat([_clean_data_] * operational_years_count, ignore_index=True)
        _all_datafields_['YEAR'] = [year for year in years for _ in range(len(_clean_data_))]

        # Add 'REGION' and 'TECHNOLOGY' columns with constant values
        _all_datafields_['REGION'] = region

        technology_name = f"{technology_prefix}{(site_sl + site_no_start - 1):02d}"
        _all_datafields_['TECHNOLOGY'] = technology_name

        # Check if the technology already exists
        if technology_name in seen_TECHNOLOGIES:
            print(f"Technology {technology_name} already exists. Recommended to revise the Site Start no. of the TECHNOLOGY.")
            continue
        else:
            print(f">>> Technology {technology_name} Created !\n *** Recommended to check the BCNexus Config file for associated datafields for technology >> '{technology_name}'")
            seen_TECHNOLOGIES.append(technology_name)  # should I save this to SETs ???

        # Assign the processed data to the nexus_timeslices dictionary
        _nexus_timeslices_[site] = _all_datafields_

    return _nexus_timeslices_

In [12]:
nexus_timeslices_SOLAR=process_timeslices_forNexus(solar_timeslices,TIMESLICES,seen_TECHNOLOGIES,"PWRSOLB",6,2035)

>>> Technology PWRSOLB06 Created !
 *** Recommended to check the BCNexus Config file for associated datafields for technology >> 'PWRSOLB06'


In [15]:
nexus_timeslices_WIND=process_timeslices_forNexus(wind_timeslices,TIMESLICES,seen_TECHNOLOGIES,"PWRWNDB",13,2035)

>>> Technology PWRWNDB13 Created !
 *** Recommended to check the BCNexus Config file for associated datafields for technology >> 'PWRWNDB13'
>>> Technology PWRWNDB14 Created !
 *** Recommended to check the BCNexus Config file for associated datafields for technology >> 'PWRWNDB14'
>>> Technology PWRWNDB15 Created !
 *** Recommended to check the BCNexus Config file for associated datafields for technology >> 'PWRWNDB15'


In [16]:
with open('new_sites_timeslice/solar_ts.pkl', 'wb') as file:
    pickle.dump(nexus_timeslices_SOLAR, file)

with open('new_sites_timeslice/wind_ts.pkl', 'wb') as file:
    pickle.dump(nexus_timeslices_WIND, file)